In [1]:
pip install sentence-transformers faiss-cpu transformers torch rouge-score PyPDF2

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 13.0 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=81940e6f2c424a01ac478a775e4a4d0c738c5d07dc8f5d0d5b600c27656debb3
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [3]:
import torch
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from rouge_score import rouge_scorer
import PyPDF2

In [4]:
text_data = """
Data Structures is a way of organizing data efficiently.
A stack follows Last In First Out principle.
A queue follows First In First Out principle.
Binary Search Trees maintain sorted order.
Hash tables provide average O(1) lookup time.
Operating Systems manage hardware resources.
Processes and threads are basic OS concepts.
Deadlock occurs when processes wait indefinitely.
"""

In [5]:
def chunk_text(text, chunk_size=100):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

chunks = chunk_text(text_data, chunk_size=20)
print(chunks)

['Data Structures is a way of organizing data efficiently. A stack follows Last In First Out principle. A queue follows', 'First In First Out principle. Binary Search Trees maintain sorted order. Hash tables provide average O(1) lookup time. Operating Systems', 'manage hardware resources. Processes and threads are basic OS concepts. Deadlock occurs when processes wait indefinitely.']


In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

print("Number of chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Number of chunks: 3
Embedding shape: (3, 384)


In [7]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS index created with", index.ntotal, "vectors.")

FAISS index created with 3 vectors.


In [14]:
def search(query, top_k=3):
  query_embedding = model.encode([query])
  distances, indices = index.search(np.array(query_embedding), top_k)

  results = [chunks[i] for i in indices[0]]
  return results

In [22]:
query = "What is a stack?"
results = search(query)

print("Query:", query)
print("\nRetrieved Chunks:")
for i, res in enumerate(results):
    print(f"\nResult {i+1}:")
    print(res)

Query: What is a stack?

Retrieved Chunks:

Result 1:
Data Structures is a way of organizing data efficiently. A stack follows Last In First Out principle. A queue follows

Result 2:
manage hardware resources. Processes and threads are basic OS concepts. Deadlock occurs when processes wait indefinitely.

Result 3:
First In First Out principle. Binary Search Trees maintain sorted order. Hash tables provide average O(1) lookup time. Operating Systems


In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Smaller LLM Loaded Successfully")

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Smaller LLM Loaded Successfully


In [25]:
def generate_answer(query, top_k=3):
    retrieved_chunks = search(query, top_k)

    context = "\n".join(retrieved_chunks)

    prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = llm.generate(**inputs, max_new_tokens=100)

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

In [26]:
query = "What is a stack?"
response = generate_answer(query)

print(response)

Data Structures
